> ## REVISED 2026-09-17 — corrected sample selection
>
> This checkpoint has been **re-run and replaced in place**. Two defects in the
> sample selection were found after the first issue (see
> `beam_metric_outliers_checkpoint`) and have been fixed here. Every number
> below is from the corrected run.
>
> **1. Calibration data was being fit as if it were beam data.** 23 of the 226
> files in this window are calibration files — the receiver was on a VNA, noise
> source or load, not on the antenna. Nothing in the mask chain was a campaign
> gate, so those samples entered the fit. They carried a **median 71.3% of the
> residual power** across the 101 channels.
>
> The exclusion is built from the **per-sample `metadata/rfswitch` stream**, not
> from the `rfswitch_dominant` column of `curation/file_state.csv`. That column
> is a file-level majority vote, and 14 of the 23 calibration files are
> *majority* `RFANT`, so a dominant-state test keeps them
> (`corr_20260717_191940Z.h5` is `{RFANT: 124, RFNON: 111, RFAMB: 3}`). Going
> per sample also *keeps* 1938 genuine on-antenna samples that a whole-file
> exclusion would have thrown away.
>
> **2. `EL_SOLUTION_GLITCH`.** `pointing_table@v1.2+45f8059` adds a bit marking
> samples where the IMU elevation solution jumped faster than the drive can
> move. Small in aggregate (~0.002 on the median) but locally decisive.
>
> Net effect on the mask: **25352 → 22969 samples** (−2200 off-antenna, −183
> glitch), and on the headline:
>
> | | before | after |
> |---|---|---|
> | median normalized RMS (corrected geometry) | 0.9029 | **0.4691** |
> | channels below 0.5 | 2 | **61** |
> | median normalized RMS (old geometry) | 0.9115 | **0.5502** |
>
> **3. Section 6 is un-blocked and rewritten.** Q8 is closed; `el` is the
> boresight zenith angle. The `|el| ≈ 180` "wrap cluster" is
> boresight-on-transmitter, not an angle-wrap artifact, and is now counted.
>
> `fit_beam_v2.py`, which generated the reports read here, lived only in `/tmp`
> and was unversioned until this revision. It now sits next to this notebook.


# D2 review checkpoint: beam-fits-v2 -- geometry stopgap fix

**Milestone:** D2, measured $A(\nu,\theta,\phi)$ + uncertainties over stated
coverage, consuming `pointing_table@v1`.

**What changed since the first (bad) run:** the first `beam-fits-v2` run held
the TX geometry (heading/alpha) fixed at `v007`'s old consensus fit, which was
calibrated against **raw motor pointing**, while feeding it
**`pointing_table@v1`-corrected** az/el for the templates. That is a
demonstrable frame mismatch (measured directly: a clean +34.8 deg elevation
offset, std 0.27 deg, plus a non-constant -30.6 deg mean / 19.8 deg std
azimuth offset from the documented slip correction) -- not a beam-quality
result. This notebook re-fits the geometry against `pointing_table@v1` itself
(orthogonal to geometer's separate joint transmitter+antenna position MCMC),
then re-runs the beam fit against the corrected geometry.

**Window:** files[-227:-1] of the campaign glob, 226 files,
`corr_20260717_185100Z.h5` .. `corr_20260718_032126Z.h5` (doc cites 227 for
the same era at `marjum-2026-07/INDEX.md:962` -- one-file boundary-inclusivity
rounding, not a different window). 64/226 dropped as comb-off; 162 files
contribute samples.

**TX-identity caveat (unresolved, carried forward unchanged):** this fit
assumes the *corrected* identity -- the 1.953125 MHz, 8-channel-locked comb is
the transmitter (rfi-analyst's rotation-modulation test). Not settled either
way by this analysis. If the identity resolves the other way, this is a
near-field self-comb map of an instrument-internal radiator, not a far-field
transmitter beam map -- same numbers, opposite physical meaning.

**Receiver-regime caveat (unresolved, carried forward unchanged):**
rf-calibrator's B7 finding places a receiver regime change (T_rx 212->539 K)
between 07-17 19:42 and 07-18 01:24 UTC, bracketing nearly this entire window.
The channel-differenced `measured_tx` quantity is largely gain-ratio-invariant
to first order, but this has not been separately verified -- a material
uncertainty on absolute amplitude scale, not on beam shape.



> **Metric convention (Aaron, 2026-09-15):** every data-vs-model comparison in
> this notebook is judged on **normalized RMS** -- residual RMS divided by the
> RMS of the data being fitted, after a single free amplitude. Correlations,
> rank correlations, peak azimuths and circular means appear only as
> *diagnostics* to explain **why** a fit behaves as it does; they are never the
> comparison itself. Two earlier conclusions in this notebook were wrong
> precisely because a diagnostic statistic was used as the metric (section 13's
> Pearson correlation, section 15's circular mean); both are marked withdrawn
> where they appear.


In [ ]:

import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import healpy as hp

sys.path.insert(0, "/mnt/data02/eigsep/eigsep_data/notebooks/arp/marjum-2026-07")
sys.path.insert(0, "/mnt/data02/eigsep/marjum-2026-07/curation")
# NOTE: /tmp is deliberately NOT on the path. fit_beam_v2.py lived only in
# /tmp until 2026-09-17 and is now version-controlled next to this
# notebook; inserting /tmp last would shadow the corrected copy with the
# stale one.

from eigsep_data.beam_mapping import compute_beam_pca, HFSSBeamSet, TransmitterGeometry
from eigsep_data.beam_mapping.diagnostics import load_v007_data
import fit_v007_pca_beam as v007
import fit_beam_v2 as v2

OUT_DIR = "/mnt/data02/eigsep/eigsep_data/notebooks/arp/marjum-2026-07"
BEAM_FILE = v2.BEAM_FILE

with open(f"{OUT_DIR}/v007_multichannel_consensus.json") as f:
    old_consensus = json.load(f)
with open(f"{OUT_DIR}/v007_multichannel_consensus_pointingv1.json") as f:
    new_consensus = json.load(f)
with open(f"{OUT_DIR}/beam_fits_v2_report.json") as f:
    old_beam = json.load(f)
with open(f"{OUT_DIR}/beam_fits_v2_pointingv1geom_report.json") as f:
    new_beam = json.load(f)

print("Loaded all four inputs OK.")


## 1. Geometry: old (raw motor pointing) vs new (`pointing_table@v1`)

In [ ]:

import pandas as pd

geom_table = pd.DataFrame({
    "old": [
        *old_consensus["heading"], old_consensus["alpha_deg"],
        old_consensus["joint_normalized_rms"], len(old_consensus["channels"]),
        len(old_consensus.get("rejected", [])),
    ],
    "new (v1 pointing)": [
        *new_consensus["heading"], new_consensus["alpha_deg"],
        new_consensus["joint_normalized_rms"], len(new_consensus["channels"]),
        len(new_consensus.get("rejected", [])),
    ],
}, index=["heading_x", "heading_y", "heading_z", "alpha_deg",
          "joint_normalized_rms", "n_channels_kept", "n_channels_rejected"])
geom_table["delta"] = geom_table["new (v1 pointing)"] - geom_table["old"]
geom_table



### 1a. The surveyed geometry -- and a retraction of the section above

**The transmitter is essentially directly below the antenna.** Taking both
positions from the archive in the same ENU frame (`transmitter_position@v2` and
`horizon_profiles`' `antenna_enu_m`, 91 m era), the transmitter is **93.0 m
below** the antenna at **6.9 m** horizontal range: a look angle of **85.8 deg
below horizontal**, near-nadir.

The machinery *can* represent this -- `ground_heading` builds `[E, N, -height]`
and `simulate_hfss_coupling` rotates that fixed heading into the antenna frame,
so nothing assumes "el = 0 points at the transmitter." But the heading actually
in use does not describe the real experiment, and **the refit performed in
section 1 made it much worse**.


In [ ]:

tx_enu = np.array(json.load(open(
    "/mnt/data02/eigsep/marjum-2026-07/curation/transmitter_position.json"
))["best_estimate_enu_m"], float)
ant_enu = np.array(json.load(open(
    "/mnt/data02/eigsep/marjum-2026-07/curation/horizon_profiles.json"
))["antenna_enu_m"], float)
delta = tx_enu - ant_enu
horiz = float(np.hypot(delta[0], delta[1]))
true_heading = delta / np.linalg.norm(delta)
print(f"antenna ENU {ant_enu}\ntransmitter ENU {tx_enu}")
print(f"antenna -> TX: dE {delta[0]:+.2f}  dN {delta[1]:+.2f}  dU {delta[2]:+.2f} m")
print(f"horizontal range {horiz:.2f} m, vertical drop {-delta[2]:.2f} m")
print(f"look angle {np.degrees(np.arctan2(-delta[2], horiz)):.1f} deg below horizontal")
print(f"surveyed heading: [{true_heading[0]:+.4f}, {true_heading[1]:+.4f}, {true_heading[2]:+.4f}]")

print()
rowsg = []
for tag, h in (("old consensus (raw motor)", np.array(old_consensus["heading"], float)),
               ("refit (pointing_table@v1)", np.array(new_consensus["heading"], float)),
               ("SURVEYED", true_heading)):
    below = np.degrees(np.arctan2(-h[2], np.hypot(h[0], h[1])))
    off = np.degrees(np.arccos(np.clip(np.dot(h, true_heading), -1, 1)))
    rowsg.append((tag, below, off))
    print(f"{tag:26s}: {below:5.1f} deg below horizontal, "
         f"{off:5.1f} deg from surveyed truth")

# theta (angle from antenna boresight to the transmitter) across the scan grid:
# does azimuth rotation move the transmitter through the beam at all?
azs = np.arange(0, 360, 5.0); els = np.arange(-180, 180, 5.0)
A, E = np.meshgrid(np.deg2rad(azs), np.deg2rad(els), indexing="ij")
for tag, h in (("refit heading", np.array(new_consensus["heading"], float)),
               ("SURVEYED heading", true_heading)):
    ca, sa = np.cos(A), -np.sin(A)
    ce, se = np.cos(E), np.sin(E)
    r2 = -se * h[1] + ce * h[2]
    th = np.degrees(np.arccos(np.clip(r2, -1, 1)))
    print(f"\n{tag}: theta from boresight spans {th.min():.1f} to {th.max():.1f} deg")
    print(f"    varying AZ at fixed el changes theta by "
         f"{np.median(th.max(axis=0)-th.min(axis=0)):.2f} deg")
    print(f"    varying EL at fixed az changes theta by "
         f"{np.median(th.max(axis=1)-th.min(axis=1)):.2f} deg")



**Retraction of section 1's conclusion.** Section 1 reported that re-fitting the
geometry against `pointing_table@v1` "worked," on the evidence that the joint
normalized RMS improved from 0.4145 to 0.3247. Measured against the survey:

| heading | below horizontal | from surveyed truth | joint RMS |
|---|---|---|---|
| old consensus (raw motor) | 78.1 deg | **16.1 deg** | 0.4145 |
| refit (`pointing_table@v1`) | 4.2 deg | **86.3 deg** | 0.3247 |
| surveyed truth | 85.8 deg | -- | -- |

**The refit improved the fit metric while moving the geometry 70 deg further
from the truth.** The old geometry was close to correct; the refit is
essentially orthogonal to it. The RMS improvement was not evidence of a better
geometry -- this is a false minimum, and section 1's conclusion is withdrawn.

**And the fit cannot see the difference.** Substituting the surveyed heading and
scanning polarization gives a best median normalized RMS of **0.9037**, against
**0.9032** for the refit heading and **0.9044** for the old one. An **86 deg
error in transmitter direction costs 0.0005 in RMS.** The transmitter direction
is therefore **not measurable from this dataset** -- the same conclusion already
reached for `alpha_deg`, and for the same reason: a fitted parameter the
objective is blind to is not a measurement.

**What the scan actually samples.** In this model the polar angle from boresight
is **independent of azimuth** (0.00 deg variation across azimuth at fixed
elevation, above): azimuth motion sweeps a *ring* of the beam at constant theta,
and only elevation changes theta. Under the surveyed geometry the scan spans
**theta = 2.7 to 177.3 deg** -- boresight to backlobe, the whole beam -- where
the refit geometry implied only 64.9 to 115.1 deg, a narrow annulus. The true
geometry is better news for D2 than the fitted one, but all of that leverage
rides on the **elevation** axis: the one flagged `EL_POST_FAILURE` across 100%
of this window. The azimuthal diversity that made this window D2's era buys
coverage in the beam's azimuthal coordinate, not in theta.


## 2. Per-channel residual RMS: old vs new geometry (consensus joint fit)

In [ ]:

old_freq = np.array(old_consensus["frequencies_mhz"])
old_rms = np.array(old_consensus["normalized_rms"])
new_freq = np.array(new_consensus["frequencies_mhz"])
new_rms = np.array(new_consensus["normalized_rms"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(old_freq, old_rms, s=16, label=f"old geometry (median {np.median(old_rms):.3f})")
axes[0].scatter(new_freq, new_rms, s=16, label=f"new geometry (median {np.median(new_rms):.3f})")
axes[0].axhline(0.60, color="k", linestyle="--", linewidth=1, label="consensus prune threshold")
axes[0].set_xlabel("Frequency [MHz]")
axes[0].set_ylabel("Normalized residual RMS")
axes[0].set_title("Joint-consensus per-channel RMS: old vs new geometry")
axes[0].legend(fontsize=8)

axes[1].hist(old_rms, bins=20, alpha=0.6, label="old geometry")
axes[1].hist(new_rms, bins=20, alpha=0.6, label="new geometry")
axes[1].set_xlabel("Normalized residual RMS")
axes[1].set_ylabel("N channels")
axes[1].set_title("Distribution")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/nb_fig_consensus_rms_comparison.png", dpi=140)
plt.show()

print(f"old joint_normalized_rms = {old_consensus['joint_normalized_rms']:.4f}")
print(f"new joint_normalized_rms = {new_consensus['joint_normalized_rms']:.4f}")


## 3. Convergence diagnostic: reduced chi-squared and fitted gain vs frequency (native output of `fit_v007_multichannel_consensus.py`'s joint optimizer)

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].semilogy(old_freq, np.array(old_consensus["channel_reduced_chisq"]), "o-", ms=3, label="old geometry")
axes[0].semilogy(new_freq, np.array(new_consensus["channel_reduced_chisq"]), "o-", ms=3, label="new geometry")
axes[0].set_ylabel("Channel reduced chi-squared")
axes[0].set_title("Per-channel reduced chi-squared (lower = better; NOTE: "
                  "these values are far above 1 in both cases, meaning "
                  "per-sample sigma is under-propagated -- expected, since "
                  "measured_sigma is a radiometer-only estimate that doesn't "
                  "include pointing/systematic error; compare shapes, not "
                  "absolute chi-sq")
axes[0].legend(fontsize=8)

axes[1].semilogy(old_freq, np.array(old_consensus["gains"]), "o-", ms=3, label="old geometry")
axes[1].semilogy(new_freq, np.array(new_consensus["gains"]), "o-", ms=3, label="new geometry")
axes[1].set_xlabel("Frequency [MHz]")
axes[1].set_ylabel("Fitted gain (consensus units)")
axes[1].set_title("Fitted gain vs frequency: does the new geometry recover a "
                  "smoother, more physical gain curve?")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/nb_fig_convergence_diagnostic.png", dpi=140)
plt.show()


## 4. Beam fit (PCA/least-squares) quality: old vs corrected geometry

In [ ]:

summary = pd.DataFrame({
    "old geometry": [
        old_beam["n_channels_fit"], old_beam["n_candidate_channels"],
        old_beam["median_normalized_rms"], old_beam["gain_fitted_vs_hfss_ratio_median"],
    ],
    "corrected geometry": [
        new_beam["n_channels_fit"], new_beam["n_candidate_channels"],
        new_beam["median_normalized_rms"], new_beam["gain_fitted_vs_hfss_ratio_median"],
    ],
}, index=["n_channels_fit", "n_candidate_channels",
          "median_normalized_rms", "gain_fitted_vs_hfss_ratio_median"])
summary


In [ ]:

old_nrms = np.array([r["normalized_rms"] for r in old_beam["channels"]])
new_nrms = np.array([r["normalized_rms"] for r in new_beam["channels"]])
old_f = np.array([r["frequency_mhz"] for r in old_beam["channels"]])
new_f = np.array([r["frequency_mhz"] for r in new_beam["channels"]])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(old_f, old_nrms, s=14, label=f"old geometry (median {np.median(old_nrms):.3f})")
ax.scatter(new_f, new_nrms, s=14, label=f"corrected geometry (median {np.median(new_nrms):.3f})")
ax.axhline(1.0, color="gray", linestyle=":", label="no better than not fitting")
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel("Normalized residual RMS (PCA beam fit)")
ax.set_title("PCA/least-squares beam-fit RMS per candidate channel")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/nb_fig_beamfit_rms_comparison.png", dpi=140)
plt.show()



## 5. Per-channel model/data/residual maps (unaveraged, az/el) -- good, bad, intermediate

The integrated-residual plots above (sections 2 and 4) summarize fit quality
as a single number per channel; they don't show *where* or *how* a fit fails.
Below are three representative channels, picked directly from the corrected
run's `normalized_rms` distribution (101 candidate channels):

- **"Good"**: channel 712 (173.83 MHz) -- **lowest** normalized RMS (0.498) of
  any candidate channel.
- **"Intermediate"**: channel 720 (175.78 MHz) -- normalized RMS 0.903,
  essentially exactly the run's median (0.9029).
- **"Bad"**: channel 432 (105.47 MHz) -- **highest** normalized RMS (1.000,
  pinned at the failed-fit ceiling -- the model explains none of the variance).

Each panel set is data, model, and residual (data - model), plotted at every
individual sample's actual (az, el) -- the same `used` mask each channel's own
fit trained on (RFI- and pointing-quality-masked; not spatially averaged or
gridded).


In [ ]:

from eigsep_data.beam_mapping import data_space_rfi_mask
from eigsep_data.beam_mapping.diagnostics import (
    channel_validity_masks, gross_power_time_flags, tx_arm_for_channel,
)

data_full = load_v007_data(v2.DATA_DIR, start=v2.FILES_SLICE[0], stop=v2.FILES_SLICE[1])
data_full = v2.attach_pointing_v1(data_full, data_full["files"])

geometry_corrected = TransmitterGeometry(new_consensus["heading"], new_consensus["alpha_deg"])
beam_full = HFSSBeamSet.from_npz(v2.BEAM_FILE)
pca_full = compute_beam_pca(beam_full, n_components=4)
templates_by_arm_full = v007._project_templates(pca_full.components, data_full, geometry_corrected)

clean_mask_full = data_space_rfi_mask(
    v2.DATA_DIR, beam_full.freqs_mhz.min(), beam_full.freqs_mhz.max(),
    files_slice=v2.FILES_SLICE, clip_sigma=5.0, min_votes=5)
clean_mask_full = clean_mask_full & v2.pointing_v1_valid_mask(data_full)

# The two corrections applied to the pipeline on 2026-09-17 (Aaron's ruling on
# the D2 review gate). Reusing v2's own functions so there is one definition of
# each mask, not a copy that can drift from the report this notebook reads.
#   - receiver_on_antenna_mask: per-sample metadata/rfswitch. 23 of the 226
#     files are calibration files; their off-antenna samples were being fit as
#     if they were beam measurements.
#   - el_solution_glitch_mask: pointing_table@v1.2's EL_SOLUTION_GLITCH bit.
_on_ant = v2.receiver_on_antenna_mask(data_full)
_no_glitch = v2.el_solution_glitch_mask(data_full)
print(f"mask: {int(clean_mask_full.sum())} -> "
      f"{int((clean_mask_full & _on_ant & _no_glitch).sum())} "
      f"(-{int((clean_mask_full & ~_on_ant).sum())} off-antenna, "
      f"-{int((clean_mask_full & _on_ant & ~_no_glitch).sum())} glitch)")
clean_mask_full = clean_mask_full & _on_ant & _no_glitch


def channel_row(channel):
    return next(r for r in new_beam["channels"] if r["channel"] == channel)


def reconstruct_channel_maps(channel):
    # Rebuild this channel's per-sample model and residual from its saved
    # fitted coefficients -- exact same math as fit_v007_pca_beam.fit_channel,
    # just evaluated instead of optimized (the optimization already happened
    # once to produce the saved report).
    row = channel_row(channel)
    arm = tx_arm_for_channel(channel)
    templates = templates_by_arm_full[arm]
    frequency_mhz = row["frequency_mhz"]

    a_hfss = v007.hfss_prior_vector(pca_full, frequency_mhz)
    gain_hfss = float(np.linalg.norm(a_hfss))
    u = a_hfss / max(gain_hfss, 1e-30)
    basis = v007._basis_from_direction(u)
    templates_rot = basis.conj().T @ templates

    params = np.r_[row["gain_fitted"],
                   np.array(row["shape_correction_real"]) + 1j * np.array(row["shape_correction_imag"])]
    model = np.abs(np.conj(params) @ templates_rot) ** 2

    y = data_full["measured_tx"][:, channel].astype(float)
    base = channel_validity_masks(data_full, [channel])[:, 0]
    gross, _, _ = gross_power_time_flags(data_full, [channel], 99.0, 5.0)
    used = base & ~gross & clean_mask_full  # exactly fit_channel's own "used" mask

    az = data_full["az_deg"][used]
    el = data_full["el_deg"][used]
    return az, el, y[used], model[used], y[used] - model[used], row

print("Setup for per-channel maps OK.")


In [ ]:
plt.rcParams["figure.dpi"] = 85

def plot_channel_maps(channel, label, reason):
    az, el, d, m, r, row = reconstruct_channel_maps(channel)
    vmin, vmax = np.percentile(d, [2, 98])
    rmax = np.percentile(np.abs(r), 98)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
    sc0 = axes[0].scatter(az, el, c=d, s=3, cmap="viridis", vmin=vmin, vmax=vmax)
    axes[0].set_title(f"Data (measured_tx), ch {channel}")
    plt.colorbar(sc0, ax=axes[0])
    sc1 = axes[1].scatter(az, el, c=m, s=3, cmap="viridis", vmin=vmin, vmax=vmax)
    axes[1].set_title(f"Model (fitted PCA beam), ch {channel}")
    plt.colorbar(sc1, ax=axes[1])
    sc2 = axes[2].scatter(az, el, c=r, s=3, cmap="RdBu_r", vmin=-rmax, vmax=rmax)
    axes[2].set_title(f"Residual (data - model), ch {channel}")
    plt.colorbar(sc2, ax=axes[2])
    for ax in axes:
        ax.set_xlabel("az [deg]")
        ax.set_ylabel("el [deg]")
    fig.suptitle(f"{label.upper()} fit example -- channel {channel}, "
                f"{row['frequency_mhz']:.2f} MHz, normalized_rms={row['normalized_rms']:.3f}\n{reason}")
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/nb_fig_channelmap_{label}.png", dpi=95)
    plt.show()


plot_channel_maps(712, "good", "lowest normalized_rms (0.498) among the 101 candidate channels")
plot_channel_maps(720, "intermediate", "closest to the run's median normalized_rms (0.903)")
plot_channel_maps(432, "bad", "highest normalized_rms (1.000, pinned at the failed-fit ceiling)")



**Reading these three together:** even the "good" channel (712) shows
structured, coherent residual patterns tracking the same az bands as the
data itself -- not white noise -- and the "bad" channel (432) shows a model
that is nearly flat while the data has real structure, i.e. the fit found
essentially no signal to attach to. The residual maps concentrate near
el ~ 0 and in specific azimuth bands across all three channels, which is a
concrete, spatially-resolved version of the same "still bad" finding in
section 4 -- and a more useful one for deciding which of the three candidate
explanations (channel screening, ridge regularization, residual geometry) is
worth chasing first.


> **REVISED 2026-09-17 — the exclusion below has been removed.** Q8 is closed.
> `geometer` established that `el` is the **boresight zenith angle**: `el = 0`
> zenith, `el = +90` horizon, `el = +180` nadir — proven independently of Aaron
> by LIDAR return character against elevation (493 real ground returns at
> `el ≈ +90` with median range 92.31 m; **zero** returns at `el ≈ −90`, i.e.
> sky; 2412 out-of-range sentinels at `el ≈ 0`, shooting across the canyon).
> The `+90 / −90` asymmetry is the sign-resolving observation.
>
> Section 1a places the transmitter essentially straight down — 85.8° below
> horizontal, i.e. near nadir. Under the now-resolved convention that puts it
> **on boresight at `|el| ≈ 180`**. The "wrap cluster" this section previously
> excluded as a suspected angle-wrap artifact is therefore the **most on-source
> pointing in the dataset**, and it is now counted in the coverage claim rather
> than footnoted out of it.

## 6. Coverage map (`pointing_table@v1.2`-only, independent of geometry/beam)

Computed on the corrected sample selection (off-antenna and
`EL_SOLUTION_GLITCH` samples removed), and reported in three parts: the total,
the on-source subset near nadir, and the rest.

In [ ]:

cov = new_beam.get("coverage_map") or old_beam.get("coverage_map")
print(json.dumps(cov, indent=2))


**Reading the coverage map.** Two caveats travel with it and are carried in
the JSON itself rather than only here:

- **The elevation zero is only bounded, not measured.** `geometer`'s MAD
  profile is flat within DEM quantisation over `el_nadir ≈ 87.5–91.5`, so the
  pointing table's el-zero offset is constrained to `|offset| ≲ 2°`. The
  zenith/nadir *sense* is settled; the zero *point* is not, to ~2°.
- **Absolute azimuth zero is still open.** The pot azimuth is a body-frame
  angle, not north-referenced, so the azimuth axis here has no absolute north
  anchor. Azimuth *coverage* is meaningful; an azimuth *bearing* read off this
  map is not.

## 7. Fitted beam vs HFSS -- shown only if the corrected fit actually converged

In [ ]:

CONVERGENCE_THRESHOLD = 0.60  # same threshold fit_consensus() itself prunes on

converged = new_beam["median_normalized_rms"] < CONVERGENCE_THRESHOLD
print(f"corrected-geometry median_normalized_rms = {new_beam['median_normalized_rms']:.4f} "
      f"({'BELOW' if converged else 'AT OR ABOVE'} the {CONVERGENCE_THRESHOLD} convergence bar)")

if not converged:
    print("\
NOT converged -- skipping the fitted-beam-vs-HFSS comparison. "
          "Showing a residual-vs-frequency plot instead so the failure mode is visible, "
          "not just a scalar.")
else:
    print("\
Converged -- reconstructing the full PCA/DPSS model to render the "
          "fitted-beam-vs-HFSS comparison maps.")


In [ ]:

def reconstruct_full_model(report, beam_file, data, n_components=4,
                           filter_half_width_ns=20.0, channel_clip_sigma=4.0):
    """Redo build_model()'s cheap DPSS-grid tail from already-fit per-channel
    rows (report['channels']) -- avoids re-running the expensive per-channel
    least_squares, which already happened once to produce the report."""
    beam = HFSSBeamSet.from_npz(beam_file)
    pca = compute_beam_pca(beam, n_components=n_components)
    rows = report["channels"]

    freqs = np.array([row["frequency_mhz"] for row in rows])
    weights = 1.0 / np.maximum(np.array([row["normalized_rms"] for row in rows]), 1e-3) ** 2
    gain_fitted = np.array([row["gain_fitted"] for row in rows])
    shape_real = np.array([row["shape_correction_real"] for row in rows])
    shape_imag = np.array([row["shape_correction_imag"] for row in rows])

    grid_lo, grid_freqs = v007._dense_frequency_grid(data, beam)
    channel_index = np.array([row["channel"] for row in rows]) - grid_lo
    grid_weight = np.zeros(grid_freqs.size)
    grid_weight[channel_index] = weights
    grid_gain_hfss = np.array([np.linalg.norm(v007.hfss_prior_vector(pca, f)) for f in grid_freqs])

    grid_gain_values = np.zeros(grid_freqs.size)
    grid_gain_values[channel_index] = gain_fitted
    grid_gain, used_gain, nterms = v007._dpss_robust_fit(
        grid_freqs, grid_weight, grid_gain_values, filter_half_width_ns,
        clip_sigma=channel_clip_sigma)
    trusted = used_gain[channel_index].copy()

    grid_shape = {"real": [], "imag": []}
    for j in range(n_components - 1):
        gv_re = np.zeros(grid_freqs.size); gv_re[channel_index] = shape_real[:, j]
        gv_im = np.zeros(grid_freqs.size); gv_im[channel_index] = shape_imag[:, j]
        model_re, used_re, _ = v007._dpss_robust_fit(
            grid_freqs, grid_weight, gv_re, filter_half_width_ns, clip_sigma=channel_clip_sigma)
        model_im, used_im, _ = v007._dpss_robust_fit(
            grid_freqs, grid_weight, gv_im, filter_half_width_ns, clip_sigma=channel_clip_sigma)
        grid_shape["real"].append(model_re)
        grid_shape["imag"].append(model_im)
        trusted &= used_re[channel_index] & used_im[channel_index]

    return {
        "geometry": report_geometry, "pca": pca, "data": data, "channels": rows,
        "freqs": freqs, "gain_fitted": gain_fitted, "gain_hfss": np.array([row["gain_hfss"] for row in rows]),
        "shape_real": shape_real, "shape_imag": shape_imag, "trusted": trusted,
        "grid_freqs": grid_freqs, "grid_gain": grid_gain, "grid_gain_hfss": grid_gain_hfss,
        "grid_shape": grid_shape, "n_components": n_components, "dpss_nterms": nterms,
    }

if converged:
    with open(f"{OUT_DIR}/v007_multichannel_consensus_pointingv1.json") as f:
        report_geometry = json.load(f)
    data = load_v007_data(v2.DATA_DIR, start=v2.FILES_SLICE[0], stop=v2.FILES_SLICE[1])
    data = v2.attach_pointing_v1(data, data["files"])
    model = reconstruct_full_model(new_beam, BEAM_FILE, data)
    v007.make_diagnostics(model, f"{OUT_DIR}/nb_fig_v2corrected")
    comparison_maps = v007.make_beam_comparison(
        model, [70.0, 130.0, 150.0, 200.0], f"{OUT_DIR}/nb_fig_v2corrected")
    print("Rendered make_diagnostics + make_beam_comparison for the corrected fit.")
else:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.scatter(new_f, new_nrms, s=14, c="C3")
    ax.axhline(CONVERGENCE_THRESHOLD, color="k", linestyle="--")
    ax.set_xlabel("Frequency [MHz]")
    ax.set_ylabel("Normalized residual RMS")
    ax.set_title("Corrected-geometry fit did NOT converge -- residual vs frequency")
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/nb_fig_v2corrected_not_converged.png", dpi=140)
    plt.show()


In [ ]:

from IPython.display import Image, display
import glob as _glob

if converged:
    for p in sorted(_glob.glob(f"{OUT_DIR}/nb_fig_v2corrected*.png")):
        display(Image(filename=p))


## 8. Per-channel HFSS agreement, single amplitude DOF (Aaron's concern 1)

Before any more PCA/joint fitting: for each of the 101 candidate channels,
independently, fit **only a scalar amplitude** `A` against the pure HFSS
prediction for that channel's frequency and (fixed) geometry -- no shape
correction, no joint re-solve, no information borrowed from any other
channel. `data ~= A * HFSS_model(az, el)`, `A` by linear least squares.

This isolates whether the PCA/joint fit's poor performance is a joint-basis
artifact (a handful of bad channels corrupting good ones through shared PCA
components) or is already present per-channel. If single-DOF and full-PCA
RMS track each other closely, the joint basis is not the cause.


In [ ]:

def single_dof_fit(channel):
    row_pca = channel_row(channel)
    arm = tx_arm_for_channel(channel)
    templates = templates_by_arm_full[arm]
    frequency_mhz = float(data_full["freqs"][channel])

    a_hfss = v007.hfss_prior_vector(pca_full, frequency_mhz)
    prior_model = np.abs(np.conj(a_hfss) @ templates) ** 2  # pure HFSS prediction, zero data-fit freedom

    y = data_full["measured_tx"][:, channel].astype(float)
    base = channel_validity_masks(data_full, [channel])[:, 0]
    gross, _, _ = gross_power_time_flags(data_full, [channel], 99.0, 5.0)
    used = base & ~gross & clean_mask_full  # identical mask to the PCA fit, for a fair comparison

    m = prior_model[used]
    d = y[used]
    A = np.sum(d * m) / max(np.sum(m ** 2), 1e-300)
    residual = d - A * m
    initial_rms = float(np.sqrt(np.mean(d ** 2)))
    normalized_rms = float(np.sqrt(np.mean(residual ** 2)) / max(initial_rms, 1e-30))
    return {
        "channel": int(channel), "frequency_mhz": frequency_mhz, "arm": int(arm),
        "amplitude_A": float(A), "n_used": int(used.sum()),
        "normalized_rms": normalized_rms, "pca_normalized_rms": row_pca["normalized_rms"],
    }


candidate_channels_list = sorted(r["channel"] for r in new_beam["channels"])
single_dof_results = [single_dof_fit(ch) for ch in candidate_channels_list]

sd_nrms = np.array([r["normalized_rms"] for r in single_dof_results])
sd_pca_nrms = np.array([r["pca_normalized_rms"] for r in single_dof_results])
sd_arm = np.array([r["arm"] for r in single_dof_results])
sd_freq = np.array([r["frequency_mhz"] for r in single_dof_results])
sd_ch = np.array([r["channel"] for r in single_dof_results])

print(f"single-DOF:  median normalized_rms = {np.median(sd_nrms):.4f}, "
     f"n agree (<0.60) = {(sd_nrms<0.6).sum()}/{len(sd_nrms)}")
print(f"full PCA fit: median normalized_rms = {np.median(sd_pca_nrms):.4f}, "
     f"n agree (<0.60) = {(sd_pca_nrms<0.6).sum()}/{len(sd_pca_nrms)}")


In [ ]:

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(sd_freq, sd_nrms, s=14, label=f"single-DOF (median {np.median(sd_nrms):.3f})")
ax.scatter(sd_freq, sd_pca_nrms, s=14, marker="x", label=f"full PCA fit (median {np.median(sd_pca_nrms):.3f})")
ax.axhline(0.6, color="k", linestyle="--", linewidth=1, label="agree/disagree threshold")
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel("Normalized residual RMS")
ax.set_title("Single-amplitude-DOF HFSS agreement vs the full PCA/joint fit -- nearly identical")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/nb_fig_single_dof_vs_pca.png", dpi=110)
plt.show()

print("Best 10 channels (single-DOF):")
for i in np.argsort(sd_nrms)[:10]:
    print(f"  ch{sd_ch[i]:4d}  arm{sd_arm[i]}  {sd_freq[i]:7.2f} MHz  nrms={sd_nrms[i]:.3f}")
print("Worst 10 channels (single-DOF):")
for i in np.argsort(sd_nrms)[-10:]:
    print(f"  ch{sd_ch[i]:4d}  arm{sd_arm[i]}  {sd_freq[i]:7.2f} MHz  nrms={sd_nrms[i]:.3f}")



**Verdict on concern 1:** the single-DOF (zero shape freedom, per-channel
independent) fit gives a median normalized RMS of essentially the same value
as the full PCA/joint fit (see the numbers printed above). **This rules out
"a handful of bad channels corrupting good ones through the joint/PCA
basis"** as the explanation -- the poor agreement is already present at the
single-channel, single-amplitude level, with no cross-channel information
sharing at all. Roughly the same ~18-19 of 101 channels agree
(normalized RMS < 0.6) under either method, and it is largely the *same*
channels in both lists (checked directly above). Whatever is wrong is a
per-channel (or per-arm/per-frequency-band) property of the data-to-HFSS
match, not an artifact of the joint fitting machinery.


## 9. Arm/polarization-parity check (Aaron's concern 2)

`tx_arm_for_channel(channel) = (channel // 8) % 2` (`diagnostics.py:261`) and
`TransmitterGeometry.field_top(arm)` (`geometry.py:176`) drive arm 0 at
`alpha_deg`, arm 1 at `alpha_deg + 90`. Two checks:

1. **Is the arm split actually doing anything mechanically?** (i.e. not a
   silent no-op where both arms get the same template.)
2. **Does per-channel fit quality correlate with arm parity?** If it does,
   that is direct evidence of a polarization-handling problem somewhere in
   the pipeline -- wrong arm assigned to a physical channel, or the fixed
   `alpha_deg` being wrong for one arm.


In [ ]:

# (1) mechanical sanity check: do the two arms' templates actually differ?
diff = np.abs(templates_by_arm_full[0] - templates_by_arm_full[1])
scale = np.abs(templates_by_arm_full[0])
identical = np.allclose(templates_by_arm_full[0], templates_by_arm_full[1])
median_rel_diff = np.median(diff[scale > 0] / scale[scale > 0])
print(f"arm0 vs arm1 templates identical? {identical}")
print(f"median relative difference: {median_rel_diff:.3f}")
print("-> the arm split is mechanically live (not a no-op); "
     "any correlation below is a real effect, not evidence the split is inert.")


In [ ]:

from scipy import stats

print(f"arm 0: n={int((sd_arm==0).sum())}  median single-DOF nrms={np.median(sd_nrms[sd_arm==0]):.4f}"
     f"  median PCA nrms={np.median(sd_pca_nrms[sd_arm==0]):.4f}")
print(f"arm 1: n={int((sd_arm==1).sum())}  median single-DOF nrms={np.median(sd_nrms[sd_arm==1]):.4f}"
     f"  median PCA nrms={np.median(sd_pca_nrms[sd_arm==1]):.4f}")

corr_pca, p_pca = stats.pointbiserialr(sd_arm, sd_pca_nrms)
corr_sd, p_sd = stats.pointbiserialr(sd_arm, sd_nrms)
print(f"point-biserial correlation(arm, PCA nrms) = {corr_pca:.3f}, p = {p_pca:.3g}")
print(f"point-biserial correlation(arm, single-DOF nrms) = {corr_sd:.3f}, p = {p_sd:.3g}")

trust = sd_nrms < 0.6
excl = sd_nrms >= 0.9
print()
print(f"trustworthy (<0.6): arm0={int(((sd_arm==0)&trust).sum())}  arm1={int(((sd_arm==1)&trust).sum())}")
print(f"excludable (>=0.9): arm0={int(((sd_arm==0)&excl).sum())}  arm1={int(((sd_arm==1)&excl).sum())}")


In [ ]:

# Control for frequency: candidate channels alternate arm every 8 raw channels,
# so each arm-0 channel has an arm-1 neighbor at nearly the same frequency.
# Compare those pairs directly, rather than pooling all channels together.
by_channel = {r["channel"]: r for r in single_dof_results}
print(f"{'ch (arm0)':>10} {'freq':>8} {'nrms':>6}   {'ch (arm1)':>10} {'freq':>8} {'nrms':>6}")
worst_arm0 = sorted([r for r in single_dof_results if r["arm"] == 0],
                    key=lambda r: -r["normalized_rms"])[:8]
for r0 in worst_arm0:
    r1 = by_channel.get(r0["channel"] + 8)
    if r1 is None or r1["arm"] != 1:
        continue
    print(f"{r0['channel']:>10} {r0['frequency_mhz']:>8.2f} {r0['normalized_rms']:>6.3f}   "
         f"{r1['channel']:>10} {r1['frequency_mhz']:>8.2f} {r1['normalized_rms']:>6.3f}")



**Verdict on concern 2:** two effects, both real, superimposed:

1. **A genuine low-frequency floor affecting both arms** -- even an arm-1
   channel next to a badly-fit arm-0 channel (same approximate frequency) is
   itself well above the agreement threshold at the low end of the band (see
   the paired comparison above: arm-1 neighbors of the worst arm-0 channels
   still sit at normalized RMS ~0.9-0.99, not ~0.5). This looks like an
   HFSS-model-vs-frequency effect (or a near-field/FM-adjacent effect at the
   low end), not an arm bug.
2. **On top of that, arm 0 is consistently and significantly worse than
   arm 1** at matched frequency: median normalized RMS 0.96 (arm 0) vs 0.75
   (arm 1), point-biserial correlation with arm parity **-0.36 to -0.37,
   p < 0.001** in both the single-DOF and full-PCA fits (same correlation
   strength in both -- so this is not a joint-fit artifact either). Of the 18
   single-DOF-trustworthy channels, 14 are arm 1; of the 51
   single-DOF-excludable channels, 36 are arm 0.

The mechanical sanity check confirms the arm/polarization split in the code
is genuinely live (arm-0 and arm-1 templates differ by a median ~170%
relative difference, not a no-op), so this is not "the split isn't being
applied at all." It is consistent with either a wrong arm-to-physical-channel
assignment, or a polarization angle (`alpha_deg`) that fits arm 1's real
response much better than arm 0's -- **not distinguished further here**; that
needs someone with the hardware polarization convention to adjudicate, not
another curve fit.


## 10. Caveat (carried forward, not new): this does not resolve the TX-identity dispute

`marjum-2026-07/INDEX.md` documents a live, unresolved dispute over whether
the whole `v007` family -- including the `alpha_deg`/heading values used as
fixed geometry throughout sections 9-10 -- measures the far-field transmitter
or the instrument's own internal self-comb. **This analysis does not
distinguish between a polarization-handling problem and a self-comb-identity
problem**: both would plausibly produce exactly this kind of structured,
arm-and-frequency-correlated residual pattern. Stated here again rather than
resolved.



## 11. Raw-HFSS panels: data / unfitted HFSS model / residual, both arms

Section 5 compared the data to the *PCA-fitted* model. These panels compare it
to the **raw HFSS model** instead: no shape correction, no joint re-solve, only
the one unavoidable amplitude scale `A` (the HFSS beam is a gain pattern; the
transmitter's absolute power is unknown, so some scale is unavoidable). The
residual is `data - A * HFSS`.

Channels are matched-frequency pairs spanning **both polarization arms**, since
the arm effect is what is under investigation:

| | arm 0 | arm 1 |
|---|---|---|
| mid-band | ch 704 (171.88 MHz) | ch 712 (173.83 MHz) |
| low-band | ch 240 (58.59 MHz) | ch 248 (60.55 MHz) |

Each figure has two rows: **all elevations** (top) and **|el| < 30 deg only**
(bottom). The second row matters because the bright bands at el ~ +/-180 are the
angle-wrap cluster already footnoted in section 6 as possibly-artifactual; on a
saturating colour scale they dominate the eye and sit at a different azimuth
from the el ~ 0 main lobe, which makes the top row easy to misread.


In [ ]:

from eigsep_data.beam_mapping.tx_model import simulate_hfss_coupling

HEADING = np.asarray(new_consensus["heading"], float)
CODE_ALPHA = float(new_consensus["alpha_deg"])
geom_code = TransmitterGeometry(HEADING, CODE_ALPHA)

chan_list = np.array(sorted(r["channel"] for r in new_beam["channels"]))
chan_arms = np.array([tx_arm_for_channel(c) for c in chan_list])
chan_freqs = np.array([float(data_full["freqs"][c]) for c in chan_list])
A_HFSS = np.array([v007.hfss_prior_vector(pca_full, f) for f in chan_freqs])
YDATA = np.array([data_full["measured_tx"][:, c].astype(float) for c in chan_list])

USED = []
for c in chan_list:
    base = channel_validity_masks(data_full, [int(c)])[:, 0]
    gross, _, _ = gross_power_time_flags(data_full, [int(c)], 99.0, 5.0)
    USED.append(base & ~gross & clean_mask_full)
USED = np.array(USED)
CH_INDEX = {int(c): i for i, c in enumerate(chan_list)}
NTIME = data_full["az_deg"].size
print(f"{len(chan_list)} channels, median n_used = {np.median(USED.sum(axis=1)):.0f}")


def hfss_models(geometry, az_offset_deg=0.0):
    # Raw HFSS predicted power per channel, each channel on its own arm.
    az = data_full["az_deg"] + az_offset_deg
    out = np.empty((len(chan_list), NTIME))
    for a in (0, 1):
        coupling, _ = simulate_hfss_coupling(
            pca_full.components, az, data_full["el_deg"], geometry,
            np.full(NTIME, a, dtype=int))
        out[chan_arms == a] = np.abs(np.conj(A_HFSS[chan_arms == a]) @ coupling) ** 2
    return out


def single_dof(model_row, y, used):
    m, d = model_row[used], y[used]
    A = np.sum(d * m) / max(np.sum(m * m), 1e-300)
    resid = d - A * m
    nrms = np.sqrt(np.mean(resid ** 2)) / max(np.sqrt(np.mean(d ** 2)), 1e-30)
    return A * m, resid, nrms


MODEL_CODE = hfss_models(geom_code)
print("raw HFSS models built at the fitted geometry")


In [ ]:
plt.rcParams["figure.dpi"] = 80

def panel(channel):
    i = CH_INDEX[channel]
    used = USED[i]
    hfss, resid, nrms = single_dof(MODEL_CODE[i], YDATA[i], used)
    az, el = data_full["az_deg"][used], data_full["el_deg"][used]
    d = YDATA[i][used]
    fig, axes = plt.subplots(2, 3, figsize=(11, 5.6))
    for row, (sel, tag) in enumerate([(np.ones(az.size, bool), "all el"),
                                      (np.abs(el) < 30.0, "|el| < 30 deg")]):
        vmin, vmax = np.percentile(d[sel], [2, 98])
        rmax = np.percentile(np.abs(resid[sel]), 98)
        s0 = axes[row, 0].scatter(az[sel], el[sel], c=d[sel], s=2, cmap="viridis",
                                 vmin=vmin, vmax=vmax)
        axes[row, 0].set_title(f"Data ({tag})")
        plt.colorbar(s0, ax=axes[row, 0])
        s1 = axes[row, 1].scatter(az[sel], el[sel], c=hfss[sel], s=2, cmap="viridis",
                                 vmin=vmin, vmax=vmax)
        axes[row, 1].set_title(f"Raw HFSS x A ({tag})")
        plt.colorbar(s1, ax=axes[row, 1])
        s2 = axes[row, 2].scatter(az[sel], el[sel], c=resid[sel], s=2, cmap="RdBu_r",
                                 vmin=-rmax, vmax=rmax)
        axes[row, 2].set_title(f"Residual = data - HFSS ({tag})")
        plt.colorbar(s2, ax=axes[row, 2])
        for ax in axes[row]:
            ax.set_xlabel("az [deg]"); ax.set_ylabel("el [deg]")
    fig.suptitle(f"ch {channel} -- arm {tx_arm_for_channel(channel)} -- "
                f"{chan_freqs[i]:.2f} MHz -- single-amplitude-DOF normalized RMS = {nrms:.3f}")
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/nb_fig_hfss_panel_ch{channel}.png", dpi=68)
    plt.show()


for _c in (704, 712, 240, 248):
    panel(_c)



## 12. Azimuth convention audit (checked in the code, not assumed)

Aaron's stated convention: *az = 0 is the orientation of the antenna's
az-polarization axis; az measured clockwise from east, passing through north as
it increases.* What the pipeline actually does:

- **Frame.** `ground_heading(east_m, north_m, height_m)` builds
  `[east, north, -height]` and `vector_to_spherical` uses `phi = atan2(y, x)`.
  So **x = East, y = North, z = Up**, and `az = 0` puts the antenna frame's
  x-axis along **East**.
- **Sense.** The production path of `simulate_hfss_coupling` defaults to
  `az_axis = (0, 0, -1)` -- a right-handed rotation about the *downward* axis,
  i.e. **clockwise seen from above** -- and its vectorized branch uses
  `sa = -sin(az)`, which matches. `scan_coverage_counts` uses the same
  expression. **These are consistent.** The generic helper `rotation_matrix`
  defaults to `az_axis = (0, 0, 1)` (the opposite sense), but the production
  path never uses that default -- worth stating explicitly, because a mismatch
  there would have been exactly the kind of bug under investigation. **Checked
  and ruled out.**
- **Internal inconsistency in the stated convention.** Viewed from above with
  north up, *clockwise* from east reaches **south** at +90 deg; it is
  *counter*-clockwise from east that passes through **north**. So "clockwise
  from east" and "passing through north as it increases" are opposite senses.
  The code matches the first half (clockwise from above).
- **The zero point is not independently registered.** `pointing_table@v1` states
  plainly: *"Azimuth zero is not tied to true north. It inherits the
  potentiometer calibration; `geometer` owns absolute registration."* So the
  premise "az = 0 is the antenna's az-polarization axis" is a claim about where
  the **potentiometer zero** physically sits, which this pipeline can neither
  confirm nor refute.

**Does that block the polarization inference?** No -- and this is the useful
part: a polarization angle measured *relative to the antenna's own axis* does
not need absolute north registration at all. The inference is blocked for a
different reason, established in section 13.



## 13. What polarization angle does the data actually imply?

`field_top(arm)` uses `alpha_deg + 90*arm`, so **arm 1 at alpha is identical to
arm 0 at alpha + 90**, and received power is invariant under a 180 deg flip of
the polarization vector. Two consequences:

1. *"The arm mapping is swapped"* and *"alpha is off by 90 deg"* are **the same
   hypothesis** in this parameterization -- the data can test it, but cannot
   distinguish which description is the cause.
2. A single scan over an **effective** polarization angle `psi` covers every
   arm/alpha combination at once.

Below: scan `psi` across the full 0-180 deg range, fit the single amplitude DOF
per channel at each `psi`, and ask what each arm's channels actually prefer.


In [ ]:

def nrms_all_channels(model):
    out = np.empty(len(chan_list))
    for i in range(len(chan_list)):
        _, _, out[i] = single_dof(model[i], YDATA[i], USED[i])
    return out


def models_at_psi(psi_deg):
    # every channel evaluated at the SAME effective polarization angle psi
    geom = TransmitterGeometry(HEADING, float(psi_deg))
    coupling, _ = simulate_hfss_coupling(
        pca_full.components, data_full["az_deg"], data_full["el_deg"], geom,
        np.zeros(NTIME, dtype=int))
    return np.abs(np.conj(A_HFSS) @ coupling) ** 2


PSI = np.arange(0.0, 180.0, 1.0)
RMS_PSI = np.array([nrms_all_channels(models_at_psi(p)) for p in PSI])
med_psi0 = np.median(RMS_PSI[:, chan_arms == 0], axis=1)
med_psi1 = np.median(RMS_PSI[:, chan_arms == 1], axis=1)
print(f"arm 0: best psi {PSI[np.argmin(med_psi0)]:5.1f} deg -> {med_psi0.min():.4f}   "
     f"(at the code's {CODE_ALPHA:.2f} deg: {np.interp(CODE_ALPHA, PSI, med_psi0):.4f})")
print(f"arm 1: best psi {PSI[np.argmin(med_psi1)]:5.1f} deg -> {med_psi1.min():.4f}   "
     f"(at the code's {(CODE_ALPHA+90)%180:.2f} deg: {np.interp((CODE_ALPHA+90)%180, PSI, med_psi1):.4f})")


In [ ]:

# THE SWAP TEST: arm-0 channels evaluated at arm-1's angle and vice versa.
i_code0, i_code1 = int(round(CODE_ALPHA)), int(round((CODE_ALPHA + 90) % 180))
idx_code = np.where(chan_arms == 0, i_code0, i_code1)
idx_swap = np.where(chan_arms == 0, i_code1, i_code0)
cols = np.arange(len(chan_list))
nrms_code_angle = RMS_PSI[idx_code, cols]
nrms_swapped = RMS_PSI[idx_swap, cols]

gap = np.median(nrms_code_angle[chan_arms == 0]) - np.median(nrms_code_angle[chan_arms == 1])
print(f"arm0 - arm1 fit-quality gap to be explained : {gap:+.4f}")
print(f"median nrms, code mapping    : {np.median(nrms_code_angle):.4f}")
print(f"median nrms, arms SWAPPED    : {np.median(nrms_swapped):.4f}")
live = nrms_code_angle < 0.8
print(f"restricted to the {live.sum()} channels with real explanatory power (nrms < 0.8):")
print(f"   median code {np.median(nrms_code_angle[live]):.4f} -> swapped {np.median(nrms_swapped[live]):.4f}")
print(f"   best improvement achieved by ANY single live channel: "
     f"{float((nrms_code_angle[live] - nrms_swapped[live]).max()):+.4f}")

# Why so insensitive? Compare model SHAPE vs AMPLITUDE between the two angles.
m_a = models_at_psi(CODE_ALPHA)[CH_INDEX[712]]
m_b = models_at_psi((CODE_ALPHA + 90) % 180)[CH_INDEX[712]]
print()
print(f"model shape correlation between alpha and alpha+90 (ch 712): "
     f"{np.corrcoef(m_a, m_b)[0,1]:.4f}")
print(f"median amplitude ratio between them            : "
     f"{np.median(m_b / np.maximum(m_a, 1e-30)):.4f}")
print("-> a 90 deg polarization change is mostly a RESCALING, which the one free")
print("   amplitude absorbs; only the ~3% shape change is available to the fit.")


In [ ]:

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.plot(PSI, med_psi0, label="arm-0 channels")
ax.plot(PSI, med_psi1, label="arm-1 channels")
ax.axvline(CODE_ALPHA, color="C0", ls=":", label=f"code alpha for arm 0 ({CODE_ALPHA:.1f} deg)")
ax.axvline((CODE_ALPHA + 90) % 180, color="C1", ls=":",
          label=f"code alpha for arm 1 ({(CODE_ALPHA+90)%180:.1f} deg)")
psi_null = np.degrees(np.arctan2(HEADING[0], -HEADING[1])) % 180
ax.axvline(psi_null, color="k", ls="--",
          label=f"predicted dipole null ({psi_null:.1f} deg)")
ax.set_xlabel("effective polarization angle psi [deg]")
ax.set_ylabel("median normalized residual RMS")
ax.set_title("Polarization angle has almost no leverage on fit quality\n"
            "(flat everywhere except the dipole-null spike)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/nb_fig_pol_scan.png", dpi=110)
plt.show()

print(f"predicted dipole-null angle from heading {HEADING.round(3)}: {psi_null:.1f} deg")
print(f"observed worst-fit angle: arm 0 {PSI[np.argmax(med_psi0)]:.0f} deg, "
     f"arm 1 {PSI[np.argmax(med_psi1)]:.0f} deg")



### 13a. Challenge to the above, and the test that settles it

Aaron's objection: *a 90 deg polarization rotation should rotate the azimuth of
maximum amplitude at el = 0 by 90 deg. That is a change in **where the peak
sits** -- a shape change by any normal definition -- not an amplitude-only
rescaling.* If right, it undercuts the mechanism claimed above.

He is right about two things, and the first version of this section was wrong:

1. **The peak really does move.** Measured below: the azimuth of maximum model
   amplitude at `|el| < 10 deg` shifts by **+85 to +95 deg** under a 90 deg
   polarization rotation, on every channel tested. Exactly as predicted.
2. **The "0.973 shape correlation" was a bad statistic.** It was a Pearson
   correlation over samples spanning ~5 orders of magnitude, so a handful of
   bright samples dominated it. Restricted to el ~ 0, the *rank* correlation is
   **0.76**, not 0.97. It should never have been quoted as a measure of angular
   structure.

What it does **not** establish is that the pattern rotates -- and that is the
difference that decides whether the refutation holds. Checked directly below by
reading the azimuth profile that generates the peak, rather than the summary
statistic computed from it.


In [ ]:

EL_ALL, AZ_ALL = data_full["el_deg"], data_full["az_deg"]


def az_profile(values, used, el_cut=10.0, binw=5.0):
    # Binned median response vs azimuth in a narrow elevation slice about el=0.
    sel = used & (np.abs(EL_ALL) < el_cut) & np.isfinite(values)
    edges = np.arange(0.0, 360.0 + binw, binw)
    idx = np.clip(np.digitize(AZ_ALL[sel], edges) - 1, 0, len(edges) - 2)
    v = values[sel]
    prof = np.full(len(edges) - 1, np.nan)
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= 5:
            prof[b] = np.median(v[m])
    return 0.5 * (edges[:-1] + edges[1:]), prof


m_code = models_at_psi(CODE_ALPHA)
m_swap = models_at_psi((CODE_ALPHA + 90) % 180)

print("azimuth of PEAK model amplitude at |el| < 10 deg:")
for ch in (712, 704, 896):
    i = CH_INDEX[ch]
    ca, pa = az_profile(m_code[i], USED[i])
    cb, pb = az_profile(m_swap[i], USED[i])
    a_pk, b_pk = ca[np.nanargmax(pa)], cb[np.nanargmax(pb)]
    shift = ((b_pk - a_pk + 180) % 360) - 180
    print(f"  ch{ch} (arm {tx_arm_for_channel(ch)}): "
         f"psi={CODE_ALPHA:.2f} -> {a_pk:6.1f} deg    "
         f"psi={(CODE_ALPHA+90)%180:.2f} -> {b_pk:6.1f} deg    shift {shift:+.0f} deg")


In [ ]:

# Read the profile that produced those peaks, instead of trusting the argmax.
i = CH_INDEX[712]
c1, p1 = az_profile(m_code[i], USED[i])
c2, p2 = az_profile(m_swap[i], USED[i])
n1, n2 = p1 / np.nanmax(p1), p2 / np.nanmax(p2)
ok = np.isfinite(n1) & np.isfinite(n2)

fig, ax = plt.subplots(figsize=(9.5, 4.4))
ax.plot(c1, n1, "o-", ms=3, label=f"psi = {CODE_ALPHA:.1f} deg")
ax.plot(c2, n2, "s-", ms=3, label=f"psi = {(CODE_ALPHA+90)%180:.1f} deg (rotated 90 deg)")
ax.axvline(c1[np.nanargmax(p1)], color="C0", ls=":", label="argmax, psi")
ax.axvline(c2[np.nanargmax(p2)], color="C1", ls=":", label="argmax, psi+90")
ax.set_xlabel("azimuth [deg]")
ax.set_ylabel("response, normalized")
ax.set_title("Two near-equal fixed lobes -- the 90 deg polarization rotation reweights\n"
            "them, it does not rotate the pattern")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/nb_fig_az_profile_lobes.png", dpi=110)
plt.show()

print(f"correlation of the two NORMALIZED azimuth profiles: "
     f"{np.corrcoef(n1[ok], n2[ok])[0,1]:.4f}")
for tag, n in ((f"psi={CODE_ALPHA:.1f}", n1), (f"psi={(CODE_ALPHA+90)%180:.1f}", n2)):
    order = np.argsort(np.nan_to_num(n))[::-1][:4]
    print(f"  {tag:12s} strongest bins: " + ", ".join(f"{c1[j]:.0f} deg:{n[j]:.2f}" for j in order))


In [ ]:

# Does the swap test survive when restricted to exactly the regime where the
# peak moves? (Also checks the worry that the residual is dominated by the
# possibly-artifactual el ~ +/-180 wrap cluster.)
wrap = np.abs(EL_ALL) > 150.0
core = np.abs(EL_ALL) < 30.0
pw = np.array([np.sum(YDATA[i][USED[i] & wrap] ** 2) for i in range(len(chan_list))])
pc = np.array([np.sum(YDATA[i][USED[i] & core] ** 2) for i in range(len(chan_list))])
print(f"share of squared data power carried by the wrap cluster: "
     f"median {np.median(pw / np.maximum(pw + pc, 1e-30)):.3f}")


def nrms_masked(model, extra):
    out = np.full(len(chan_list), np.nan)
    for i in range(len(chan_list)):
        u = USED[i] & extra
        if u.sum() < 50:
            continue
        m, d = model[i, u], YDATA[i, u]
        A = np.sum(d * m) / max(np.sum(m * m), 1e-300)
        r = d - A * m
        out[i] = np.sqrt(np.mean(r ** 2)) / max(np.sqrt(np.mean(d ** 2)), 1e-30)
    return out


print()
print(f"{'samples used':28s} {'arm0 delta':>11s} {'arm1 delta':>11s}")
for tag, mask in (("all used (original test)", np.ones(EL_ALL.size, bool)),
                  ("core, |el|<30", core),
                  ("el~0 only, |el|<10", np.abs(EL_ALL) < 10.0)):
    a, b = nrms_masked(m_code, mask), nrms_masked(m_swap, mask)
    d0 = np.nanmedian(b[chan_arms == 0]) - np.nanmedian(a[chan_arms == 0])
    d1 = np.nanmedian(a[chan_arms == 1]) - np.nanmedian(b[chan_arms == 1])
    print(f"{tag:28s} {d0:+11.4f} {d1:+11.4f}")
print()
print("against the arm-0-vs-arm-1 gap of 0.208")



**Reconciliation, and the corrected mechanism.**

Both statements are true at once, and the profile above shows why:

- The azimuth profile at el ~ 0 consists of **two near-equal lobes, fixed at
  az ~ 252 deg and ~ 338 deg**. At `psi = 10.07 deg` the 252 deg lobe leads by
  about 2%; at `psi = 100.07 deg` the 338 deg lobe leads. The polarization
  rotation **reweights two stationary lobes by a few percent** -- it does not
  rotate the pattern.
- Because the lobes sit 86 deg apart and are within ~2% of each other,
  `argmax` flips from one to the other and *reports* an 85-95 deg shift. The
  argmax is a discontinuous statistic and is fragile exactly here.
- Measured properly, the **normalized azimuth profiles correlate at 0.9935**.
  The angular structure is essentially unchanged -- which is what the fit
  responds to.

So the earlier conclusion stands, but the earlier *explanation* was wrong and is
withdrawn. **Corrected mechanism:** a 90 deg polarization rotation modestly
reweights two fixed lobes ~86 deg apart, leaving the angular profile nearly
unchanged; the fit is insensitive to polarization for that reason -- **not**
because polarization reduces to an amplitude rescaling, and **not** because the
model is blind to angular position.

**And the refutation was re-tested in the regime the objection targets.**
Restricting the swap test to `|el| < 10 deg` -- exactly where the peak moves --
still gives a change of ~0.002 against the 0.208 gap. The wrap cluster carries
45% of the squared power but excluding it changes nothing.

**A caveat that comes with the corrected mechanism and did not come with the
old one:** the near-equality of those two lobes may be specific to this
transmitter geometry (the heading sits only ~4 deg below horizontal). This
degeneracy should not be assumed to hold for other configurations without
re-checking.

*Method note:* both the original claim and the objection rested on summary
statistics that turned out not to mean what they appeared to -- a Pearson
correlation inflated by dynamic range, and an argmax flipping between
near-equal lobes. The profile plot settled it. Same standing rule that caught
the per-camera Fisher attribution: **a derived quantity is evidence only once
you have checked what generated it.**



## 14. Is it an azimuth-registration offset instead?

The ~90 deg offset was an *observation*; "polarization mismatch" was one
*interpretation*, now refuted. The other natural interpretation of the same
observation is that the azimuth zero point is wrong. Test: apply a global
offset to the pointing azimuth, scan it over the full circle, and see whether
anywhere -- especially +/-90 deg -- fits better than zero.

Note the built-in caveat: the heading was *fitted* at the current convention, so
it has already absorbed any global azimuth offset; zero should be locally
optimal by construction. What the scan is really testing is whether a **deeper
minimum exists elsewhere** that the joint fit missed -- i.e. the false-minimum
question -- and specifically whether one sits near 90 deg.


In [ ]:

DAZ = np.arange(0.0, 360.0, 2.0)
RMS_AZ = np.array([nrms_all_channels(hfss_models(geom_code, d)) for d in DAZ])
med_az0 = np.median(RMS_AZ[:, chan_arms == 0], axis=1)
med_az1 = np.median(RMS_AZ[:, chan_arms == 1], axis=1)

print(f"arm 0: best offset {DAZ[np.argmin(med_az0)]:5.1f} deg -> {med_az0.min():.4f}  "
     f"(at 0 deg: {med_az0[0]:.4f}, improvement {med_az0[0]-med_az0.min():+.4f})")
print(f"arm 1: best offset {DAZ[np.argmin(med_az1)]:5.1f} deg -> {med_az1.min():.4f}  "
     f"(at 0 deg: {med_az1[0]:.4f}, improvement {med_az1[0]-med_az1.min():+.4f})")
print()
for d in (0, 90, 180, 270):
    i = int(d / 2)
    print(f"  offset {d:3d} deg : arm0 {med_az0[i]:.4f}   arm1 {med_az1[i]:.4f}")

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.plot(DAZ, med_az0, label="arm-0 channels")
ax.plot(DAZ, med_az1, label="arm-1 channels")
ax.axvline(90, color="k", ls="--", label="the suspected 90 deg offset")
ax.set_xlabel("global azimuth offset applied to the pointing [deg]")
ax.set_ylabel("median normalized residual RMS")
ax.set_title("Azimuth-offset scan: 90 deg makes the fit WORSE, not better;\n"
            "response is 180-deg periodic (two-fold beam symmetry)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/nb_fig_az_scan.png", dpi=110)
plt.show()



**Verdict on the azimuth-offset hypothesis: also refuted -- and one structural
discovery.**

- A **90 deg** azimuth offset makes the fit *worse*, not better (arm 1:
  0.748 -> 0.802). The best offset found anywhere in the full circle buys 0.012
  for arm 0 and 0.002 for arm 1 -- again far too small to explain a 0.21 gap.
- **The azimuth response is 180-deg periodic**: offsets of 0 and 180 deg give
  identical results to four decimal places, as do 90 and 270. That is the
  bowtie beam's two-fold symmetry, and it means **the fitted transmitter heading
  azimuth is only ever determined modulo 180 deg** -- a real, permanent
  degeneracy of this measurement that should be recorded wherever that heading
  is quoted. It also means the fit cannot tell the transmitter's direction from
  its antipode.



## 15. Do the two arms carry different structure? (RMS test)

**This section previously reported that the two arms' power concentrates at the
same azimuth "to within 2.4 deg." That was wrong and is withdrawn.** It used a
power-weighted circular mean -- a *centroid* -- over a profile that turns out to
be multi-lobed and, between arms, nearly complementary. Averaging two opposed
lobe systems produces a number with no meaning. Aaron's objection that the
*peak* is the right statistic, not the centroid, was correct.

Re-done here as an **RMS test**, which is the metric this analysis is judged on.
The question in RMS form: *is a channel's azimuth profile better explained by an
empirical template built from its own arm, or from the other arm?* Templates are
built **leave-one-out** -- the channel under test never contributes to its own
template -- so the comparison is not circular. The HFSS model is scored on the
same binned quantity for a like-for-like comparison.


In [ ]:

BINW, ELCUT = 5.0, 10.0
EDGES = np.arange(0.0, 360.0 + BINW, BINW)
NB = len(EDGES) - 1


def binned_az_profile(values, used):
    # median response per azimuth bin, in a narrow elevation slice about el=0
    sel = used & (np.abs(EL_ALL) < ELCUT) & np.isfinite(values)
    idx = np.clip(np.digitize(AZ_ALL[sel], EDGES) - 1, 0, NB - 1)
    v = values[sel]
    pr = np.full(NB, np.nan)
    for b in range(NB):
        m = idx == b
        if m.sum() >= 5:
            pr[b] = np.median(v[m])
    return pr


PROF = {int(c): binned_az_profile(YDATA[CH_INDEX[int(c)]], USED[CH_INDEX[int(c)]])
        for c in chan_list}
MODEL_PROF = {int(c): binned_az_profile(MODEL_CODE[CH_INDEX[int(c)]], USED[CH_INDEX[int(c)]])
              for c in chan_list}


def template_rms(target, template):
    # single free amplitude, then normalized RMS -- same objective used throughout
    ok = np.isfinite(target) & np.isfinite(template)
    if ok.sum() < 10:
        return np.nan
    t, m = target[ok], template[ok]
    A = np.sum(t * m) / max(np.sum(m * m), 1e-300)
    r = t - A * m
    return float(np.sqrt(np.mean(r ** 2)) / max(np.sqrt(np.mean(t ** 2)), 1e-30))


def stacked_template(exclude, arm):
    rows = []
    for c in chan_list:
        if int(c) == int(exclude) or tx_arm_for_channel(int(c)) != arm:
            continue
        p = PROF[int(c)]
        mx = np.nanmax(p)
        if np.isfinite(mx) and mx > 0:
            rows.append(p / mx)
    return np.nanmean(np.array(rows), axis=0)


rows = []
for c in chan_list:
    a = tx_arm_for_channel(int(c))
    rows.append(dict(channel=int(c), arm=a,
                     own=template_rms(PROF[int(c)], stacked_template(c, a)),
                     other=template_rms(PROF[int(c)], stacked_template(c, 1 - a)),
                     hfss=template_rms(PROF[int(c)], MODEL_PROF[int(c)])))

for a in (0, 1):
    sub = [r for r in rows if r["arm"] == a and np.isfinite(r["own"])]
    own = np.array([r["own"] for r in sub]); oth = np.array([r["other"] for r in sub])
    hf = np.array([r["hfss"] for r in sub])
    print(f"arm {a} (n={len(sub)}): median normalized RMS of the binned azimuth profile vs")
    print(f"    own-arm empirical template : {np.median(own):.4f}")
    print(f"    other-arm template         : {np.median(oth):.4f}")
    print(f"    HFSS physical model        : {np.median(hf):.4f}")
    print(f"    own-arm beats other-arm on {int((own<oth).sum())}/{len(sub)} channels;"
         f" beats HFSS on {int((own<hf).sum())}/{len(sub)}")

allr = [r for r in rows if np.isfinite(r["own"])]
o = np.array([r["own"] for r in allr]); t = np.array([r["other"] for r in allr])
h = np.array([r["hfss"] for r in allr])
print()
print(f"ALL {len(allr)} channels: own {np.median(o):.4f}   other {np.median(t):.4f}   "
     f"HFSS {np.median(h):.4f}")
print(f"the blind own-arm template beats the physical model on "
     f"{int((o<h).sum())}/{len(allr)} channels")


In [ ]:

# The two arms' stacked profiles, which is what those RMS numbers are measuring.
t0 = stacked_template(-1, 0)
t1 = stacked_template(-1, 1)
ok = np.isfinite(t0) & np.isfinite(t1)
centers = 0.5 * (EDGES[:-1] + EDGES[1:])

fig, ax = plt.subplots(figsize=(9.5, 4.4))
ax.plot(centers, t0, "o-", ms=3, label="arm 0 stacked profile")
ax.plot(centers, t1, "s-", ms=3, label="arm 1 stacked profile")
ax.set_xlabel("azimuth [deg]")
ax.set_ylabel("normalized response at |el|<10 deg")
ax.set_title("The two polarization arms carry opposite azimuth structure\n"
            "(where one peaks, the other nulls)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/nb_fig_arm_profiles.png", dpi=110)
plt.show()

print(f"diagnostic only (not the metric): correlation between the two arms' "
     f"stacked profiles = {np.corrcoef(t0[ok], t1[ok])[0,1]:+.4f}")



**Verdict, in RMS.**

- A channel's azimuth profile is matched to **0.24** normalized RMS by a
  template built from *other channels of its own arm*, and to only **0.96** by
  the *other arm's* template. The two arms are nearly orthogonal in their
  azimuth structure.
- The **HFSS physical model scores 0.67** on the same quantity, and is **beaten
  by the blind empirical template on 93 of 101 channels**. Arm 1 is the extreme
  case: its own-arm template reaches **0.07**, i.e. the azimuth profile is
  essentially identical across all 50 arm-1 channels.
- So the polarization alternation is real, strong and highly repeatable, and
  **the model does not reproduce it**. This is the RMS-based statement of what
  section 13 detected and mis-explained.

**Important scale caveat:** these RMS values are computed on *binned* azimuth
profiles at |el|<10 deg, so they are not comparable to the per-sample
normalized RMS (~0.90) quoted elsewhere -- binning averages noise down. Only the
*relative* comparison (own vs other vs HFSS, all on the same binned quantity)
is meaningful.

**This also explains the arm-0/arm-1 fit-quality gap** that has been open since
section 9. The model produces essentially one azimuth profile for both arms
(section 13a); the data has two opposite ones. The model therefore matches
whichever arm it happens to resemble -- arm 1 -- and fails the other. The gap
was never a mysterious property of alternate tones; it is the polarization
signature, seen through a model that cannot represent it.


## 16. Decision requested

**The framing of this section has changed materially since the last issue, and
the previous verdict is withdrawn in place.** It read *"D2 has no defensible
beam measurement."* That conclusion was reached on a sample set in which
calibration data — receiver on a load, not the antenna — supplied a median
71.3% of the residual power. With that removed the fit is far better than this
notebook previously reported.

**What the corrected run shows.** Median normalized RMS against the corrected
geometry is **0.4691**, down from 0.9029, with **61 of 101 channels below
0.5** where previously there were 2. On the single-amplitude-DOF test the
median is **0.5176**, with 65/101 channels agreeing to better than 0.60. This
is a real fit, not a null result. The blunt "no measurement here" verdict was
an artifact of the selection defect.

**What did *not* change, and this is the important part.** The arm-structure
finding is completely unmoved:

| | before | after |
|---|---|---|
| own-arm empirical template | 0.242 | **0.2416** |
| other-arm template | 0.956 | **0.9556** |
| HFSS physical model | 0.670 | **0.6697** |
| blind template beats physical model on | 93/101 | **93/101** |
| arm-to-arm profile correlation | −0.92 | **−0.9211** |

That robustness is itself evidence rather than a coincidence. Section 15 works
on per-azimuth-bin **medians** within `|el| < 10°`; that slice is 10.8%
contaminated, and a median absorbs a 10.8% minority. So the arm result never
depended on the bad samples — it is a property of the data, and the physical
model still fails to reproduce it while a blind empirical template succeeds.

The arm-0/arm-1 gap also survives: median PCA normalized RMS is **0.5505** for
arm 0 against **0.4539** for arm 1 (point-biserial correlation −0.353,
p = 3e-4).

**So the D2 position is now:** the beam fit is usable on a majority of
channels, but the forward model still cannot represent a strong, repeatable
polarization structure that a blind per-arm template captures at 0.24. Three
independent refutations from the previous issue (frame mismatch, arm/pol angle
swap, azimuth registration) are unaffected — each was refuted by ~2 orders of
magnitude, far beyond the factor this correction moves.

**Still unresolved, carried forward unchanged:**

- **The transmitter direction is not measurable from this dataset.** The fitted
  heading sits 86.3° from the surveyed direction and substituting the surveyed
  one changes the median by ~0.0005 (section 1a). The earlier geometry refit
  *improved* RMS while moving 70° away from truth — a false minimum. Nothing
  here changes that; geometry should be taken from the survey, not fitted.
- **TX identity** (section 10). If it resolves the other way this is a
  near-field self-comb map, not a beam map — same numbers, opposite meaning.
- **Receiver regime change** bracketing the window: amplitude scale, not shape.
- **Elevation zero** is bounded to `|offset| ≲ 2°`, and **absolute azimuth zero
  is still open** — the coverage map has no north anchor.

**Recommended next steps** (for Aaron / `experimental-strategist` to rank, not
for me to self-assign):

1. **Re-derive geometry from the survey rather than fitting it**, and re-score.
   This is now the largest remaining known-wrong input.
2. **Treat the per-arm empirical templates as the benchmark** any physical
   model must beat. The 0.24 / 0.67 gap is the real modelling target, and it is
   now the dominant residual rather than one defect among several.
3. Hand the calibration-window leak and the per-sample `rfswitch` mask to
   `data-archivist` as durable facts about the data, so no other analysis
   repeats this.

**STOPPED AT REVIEW GATE — awaiting Aaron's approval.**